# 2장 — Working with text dataRaschka, *Build a LLM (From Scratch)* 2.2 ~ 2.4절.세션 02에서 루트에 흩어져 있던 `verdict.py` / `tokenizer.py`를 이 노트북으로 합쳤다.남길 코드는 `src/tokenizer.py`에 있다.**주의:** `data/the-verdict.txt`는 현재 손상되어 있다 (줄 끝 백슬래시 82개).토큰 수가 책의 4,690이 아니라 4,772로 나온다. 재취득 전까지는 책과 숫자가 어긋난다.자세한 내용은 `docs/LLM_project_session_02.md` §2-A.

## 2.2 텍스트를 토큰으로 나누기

In [1]:
import re
import sys

sys.path.append("..")

CORPUS = "../data/Annabel_Lee.txt"

with open(CORPUS, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 1539
It was many and many a year ago,
   In a kingdom by the sea,
That a maiden there lived whom you may


In [2]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("Number of tokens:", len(preprocessed))
print(preprocessed[:30])

Number of tokens: 330
['It', 'was', 'many', 'and', 'many', 'a', 'year', 'ago', ',', 'In', 'a', 'kingdom', 'by', 'the', 'sea', ',', 'That', 'a', 'maiden', 'there', 'lived', 'whom', 'you', 'may', 'know', 'By', 'the', 'name', 'of', 'Annabel']


## 2.3 토큰을 토큰 ID로 변환

In [3]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size:", vocab_size)

vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

Vocabulary size: 142
('!', 0)
('(', 1)
(')', 2)
(',', 3)
('.', 4)
(';', 5)
('A', 6)
('And', 7)
('Annabel', 8)
('But', 9)
('By', 10)
('Can', 11)
('Chilling', 12)
('Coveted', 13)
('For', 14)
('Heaven', 15)
('I', 16)
('In', 17)
('It', 18)
('Lee', 19)
('Lee—', 20)
('My', 21)
('Nor', 22)
('Of', 23)
('So', 24)
('Than', 25)
('That', 26)
('The', 27)
('To', 28)
('Went', 29)
('With', 30)
('Yes', 31)
('a', 32)
('above', 33)
('ago', 34)
('all', 35)
('and', 36)
('angels', 37)
('as', 38)
('away', 39)
('be', 40)
('beams', 41)
('beautiful', 42)
('blew', 43)
('bore', 44)
('bride', 45)
('bright', 46)
('bringing', 47)
('but', 48)
('by', 49)
('came', 50)


### SimpleTokenizerV1책을 따라 직접 타이핑한 원본. 정리된 판은 `src/tokenizer.py`에 있다.

In [4]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [5]:
tokenizer = SimpleTokenizerV1(vocab)

text = "In a kingdom by the sea, I loved Annabel Lee."
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[17, 32, 72, 49, 119, 105, 3, 16, 80, 8, 19, 4]
In a kingdom by the sea, I loved Annabel Lee.


## 2.4 특수 문맥 토큰 — 여기서 V1이 무너진다아래 셀은 **의도적으로 실패한다.** 어휘는 *Annabel Lee*에서 만들었는데테스트 문장은 *the-verdict*에서 가져왔다. 어휘에 없는 `"`에서 `KeyError`가 난다.이것이 책이 `<|unk|>`와 `<|endoftext|>`를 도입하는 이유다.V1은 "모르는 단어"라는 개념 자체가 없다 — 어휘 밖의 것을 만나면 표현할 방법이 없어 죽는다.**다음 작업:** `SimpleTokenizerV2`를 만들어 이 셀이 통과하게 한다.

In [6]:
text = '''"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride.'''

try:
    print(tokenizer.encode(text))
except KeyError as e:
    print("KeyError:", e, "  <- 어휘에 없는 토큰")

KeyError: '"'   <- 어휘에 없는 토큰


## 모듈에서 불러 쓰기`src/tokenizer.py`로 옮긴 판. 정규식이 세 군데 중복되어 있던 것을`preprocess()` 하나로 합쳤다.

In [7]:
from src.tokenizer import SimpleTokenizerV1 as Tokenizer, build_vocab, preprocess

vocab2 = build_vocab(raw_text)
tok = Tokenizer(vocab2)

text = "In a kingdom by the sea, I loved Annabel Lee."
assert tok.decode(tok.encode(text)) == text
print("왕복 성공:", tok.decode(tok.encode(text)))

왕복 성공: In a kingdom by the sea, I loved Annabel Lee.


## 관찰 — 구분자가 모르는 문자책의 정규식은 `--`(하이픈 두 개)는 알지만 `—`(em dash, U+2014)는 모른다.그래서 `love—`가 통째로 하나의 토큰이 된다.모델 입장에서 `love`와 `love—`는 **아무 관계도 없는 두 단어**다.

In [8]:
odd = [w for w in sorted(vocab2) if any(ord(c) > 127 for c in w)]
print(odd)
print("\n'love' in vocab:", "love" in vocab2)
print("'love—' in vocab:", "love\u2014" in vocab2)

['Lee—', 'darling—my', 'love—', 'me—', 'sea—', 'we—', 'wingèd', '—that']

'love' in vocab: True
'love—' in vocab: True


### 직접 해 볼 것- `tiktoken`으로 `love—`, `wingèd`를 인코딩해서 몇 조각으로 쪼개지는지 확인 (2.5절 BPE)- 같은 문장의 한국어판/영어판 토큰 수 비교 — 세션 01의 "2~3배" 주장을 실측으로 검증- 정규식에서 `--`를 빼고 돌려서 어휘가 어떻게 변하는지 관찰